In [1]:
import duckdb
DATA_DIR = "../../data"
duckdb.sql(f"CREATE VIEW traces AS SELECT * FROM read_csv_auto('{DATA_DIR}/learning_traces_sample.csv')")

In [2]:
duckdb.sql("SELECT COUNT(DISTINCT user_id) FROM traces").show()

┌─────────────────────────┐
│ count(DISTINCT user_id) │
│          int64          │
├─────────────────────────┤
│                   75294 │
└─────────────────────────┘



In [3]:
duckdb.sql("SELECT * FROM traces LIMIT 5").show()

┌─────────────────────┬─────────┬─────────────┬───────────────────┬──────────────┬─────────┬─────────┬──────────────┬──────────┬──────────────┬─────────────────┬──────────────┬─────────────────┬───────────┬──────────────────────────────────┐
│    practice_time    │ user_id │ ui_language │ learning_language │ surface_form │  lemma  │   pos   │ grammar_tags │ lag_days │ history_seen │ history_correct │ session_seen │ session_correct │ p_recall  │            lexeme_id             │
│      timestamp      │ varchar │   varchar   │      varchar      │   varchar    │ varchar │ varchar │   varchar    │  double  │    int64     │      int64      │    int64     │      int64      │  double   │             varchar              │
├─────────────────────┼─────────┼─────────────┼───────────────────┼──────────────┼─────────┼─────────┼──────────────┼──────────┼──────────────┼─────────────────┼──────────────┼─────────────────┼───────────┼──────────────────────────────────┤
│ 2013-02-28 20:11:37 │ u:FO    

In [4]:
duckdb.sql("""
CREATE TABLE user_sample AS
SELECT user_id FROM (SELECT DISTINCT user_id FROM traces)
ORDER BY random() LIMIT 2500
""")

duckdb.sql("""
CREATE TABLE duolingo_flagship AS
SELECT t.*
FROM traces t
WHERE t.user_id IN (SELECT user_id FROM user_sample)
""")

In [5]:
duckdb.sql("""
SELECT COUNT(*)
FROM duolingo_flagship
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        16382 │
└──────────────┘



In [6]:
duckdb.sql("""
SELECT COUNT(DISTINCT user_id)
FROM duolingo_flagship
""").show()

┌─────────────────────────┐
│ count(DISTINCT user_id) │
│          int64          │
├─────────────────────────┤
│                    2500 │
└─────────────────────────┘



In [7]:
duckdb.sql("""
SELECT * FROM duolingo_flagship LIMIT 5
""").show()

┌─────────────────────┬─────────┬─────────────┬───────────────────┬──────────────┬─────────┬─────────┬───────────────┬──────────┬──────────────┬─────────────────┬──────────────┬─────────────────┬──────────┬──────────────────────────────────┐
│    practice_time    │ user_id │ ui_language │ learning_language │ surface_form │  lemma  │   pos   │ grammar_tags  │ lag_days │ history_seen │ history_correct │ session_seen │ session_correct │ p_recall │            lexeme_id             │
│      timestamp      │ varchar │   varchar   │      varchar      │   varchar    │ varchar │ varchar │    varchar    │  double  │    int64     │      int64      │    int64     │      int64      │  double  │             varchar              │
├─────────────────────┼─────────┼─────────────┼───────────────────┼──────────────┼─────────┼─────────┼───────────────┼──────────┼──────────────┼─────────────────┼──────────────┼─────────────────┼──────────┼──────────────────────────────────┤
│ 2013-03-11 14:15:03 │ u:f5Zo  